In [14]:
import sqlite3
import json
import pathlib
import pandas as pd # Utiliser pour visualiser la db il faudra le supprimer à la fin
import re

In [2]:
conn = sqlite3.connect('infectio_git.db')
cursor = conn.cursor()

à clarifier : le nombre max j'ai mis 20 mais à voir
sqlite apparemment ne met pas de limite de caractere si je met TEXT ou autre ?


## Tables/entités principales

In [11]:
cursor.execute("""CREATE TABLE IF NOT EXISTS Modele(
               formalisme TEXT,
               echelle TEXT,
               DOI TEXT, 
               annee INTEGER,
               logiciel TEXT,
               CONSTRAINT num_doi PRIMARY KEY(DOI) );
               """)

cursor.execute("""CREATE TABLE IF NOT EXISTS Maladie(
               nom TEXT,
               DOID_MONDO TEXT );
               """)

cursor.execute("""CREATE TABLE IF NOT EXISTS Pathogene(
               espece TEXT,
               identifiant_taxonomique TEXT,
               classe TEXT, 
               annee INTEGER );
               """)

cursor.execute("""CREATE TABLE IF NOT EXISTS Hote(
               espece TEXT,
               in_vitro_vivo TEXT);
               """)

# cursor.execute("""CREATE TABLE IF NOT EXISTS Tissu ?

cursor.execute("""CREATE TABLE IF NOT EXISTS Types_cellulaires(
               Ontologie_cellulaire TEXT);
               """)

## Tables de liaison

In [13]:
cursor.execute("""CREATE TABLE IF NOT EXISTS modele_maladie (
            modele_doi TEXT,
            maladie_doid TEXT,
            PRIMARY KEY (modele_doi, maladie_doid),
            FOREIGN KEY (modele_doi) REFERENCES Modele(DOI),
            FOREIGN KEY (maladie_doid) REFERENCES Maladie(DOID_MONDO));
            """)


cursor.execute("""CREATE TABLE IF NOT EXISTS modele_pathogene (
            modele_doi TEXT,
            pathogene_taxonomie TEXT,
            PRIMARY KEY (modele_doi, pathogene_taxonomie),
            FOREIGN KEY (modele_doi) REFERENCES Modele(DOI),
            FOREIGN KEY (pathogene_taxonomie) REFERENCES Pathogene(identifiant_taxonomique));
            """)

# modèle tissus ?

cursor.execute("""CREATE TABLE IF NOT EXISTS modele_types_cellulaires (
            modele_doi TEXT,
            typecell_ontologie TEXT,
            PRIMARY KEY (modele_doi, typecell_ontologie),
            FOREIGN KEY (modele_doi) REFERENCES Modele(DOI),
            FOREIGN KEY (typecell_ontologie) REFERENCES Types_cellulaires(Ontologie_cellulaire));
            """)

# modele - voie métabolique ????
# modele - jeu de donnees ????

## Création des fonctions pour le parsing

In [ ]:
with open('test-data.json', 'r', encoding='utf-8') as f:
    donnees = json.load(f)
     

def match_DOID(url):
  res = re.search(r"doi :+", url)
  if res:
    resultat = res.group()
    return(resultat)

def recherche_DOID(query) :
  DOID_list=[]
  graphe = donnees["graphs"][0]["nodes"]

  for results in graphe:
      # vérification si la clé 'lbl' existe et correspond à la query
      if "lbl" in results and query.lower() in results["lbl"].lower():
          # extraction l'identifiant via l'URL présente dans 'id'
          doid = match_DOID(results["id"])
          if doid:
              DOID_list.append(f"{results["lbl"]} : {doid}")

  return DOID_list
     


In [17]:
def extractDOID(fichier_json):
    res = re.search(r'doi: +', fichier_json)
    if res : 
        return res

print(extractDOID("test_data.json"))
    

None


In [8]:
cursor.execute("""INSERT INTO Modèle VALUES ("test", "test2", "XX565X", 2839, "test3");""")

In [9]:
df = pd.read_sql_query("SELECT * FROM Modèle", conn)
df 

,formalisme,echelle,DOI,annee,logiciel
0,test,test2,XX565X,2839,test3
